In [11]:
import sys 
from pathlib import Path

repo_root = Path.cwd().parents[1]
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

In [12]:
import numpy as np
import os
import pandas as pd
from pathlib import Path
from tqdm.autonotebook import tqdm
import statsmodels.api as sm
import warnings
import random
from IPython.display import display, HTML
np.random.seed(42)
random.seed(42)
tqdm.pandas()

In [13]:
try:
    lclid_acorn_map = pd.read_pickle("/Users/Soham/Documents/FALL2026/MTFS_textbook_work/data/london_smart_meters/preprocessed/london_smart_meters_lclid_acorn_map.pkl")
except FileNotFoundError:
    display(
        HTML(
            '''
                <div class="alert alert-block alert warning">
                <b>Warning!<b> File not found. Please make sure you have run 02-Preprocessing London Smart Meter Dataset.ipynb in Chapter02
                </div>
            '''
        )
    )

In [14]:
affluent_households = lclid_acorn_map.loc[lclid_acorn_map.Acorn_grouped=="Affluent", ["LCLid",'file']]
adversity_households = lclid_acorn_map.loc[lclid_acorn_map.Acorn_grouped=="Adversity", ["LCLid",'file']]
comfortable_households = lclid_acorn_map.loc[lclid_acorn_map.Acorn_grouped=="Comfortable", ["LCLid",'file']]

In [15]:
selected_households = pd.concat(
    [affluent_households.sample(n=100, random_state=76),
     adversity_households.sample(n=100, random_state=76),
     comfortable_households.sample(n=100, random_state=76)]
    )
selected_households['block'] = selected_households.file.str.split("_", expand=True).iloc[:,1].astype(int)

In [16]:
# extracting the paths to the different blocks and extracting the starting and ending blocks
path_blocks = [
    (p, *list(map(int, p.name.split("_")[5].split(".")[0].split("-"))))
    for p in Path("/Users/Soham/Documents/FALL2026/MTFS_textbook_work/data/london_smart_meters/preprocessed").glob(
        "london_smart_meters_merged_block*"
    )
]

In [17]:
household_df_l = []
for path, start_b, end_b in tqdm(path_blocks):
    block_df = pd.read_parquet(path)
    mask = selected_households['block'].between(start_b, end_b)
    lclids = selected_households.loc[mask, 'LCLid']
    household_df_l.append(block_df.loc[block_df.LCLid.isin(lclids)])

100%|██████████| 14/14 [00:15<00:00,  1.10s/it]


In [18]:
block_df = pd.concat(household_df_l)
del household_df_l
block_df.head()

,LCLid,start_timestamp,frequency,energy_consumption,series_length,stdorToU,Acorn,Acorn_grouped,file,holidays,...,windBearing,temperature,dewPoint,pressure,apparentTemperature,windSpeed,precipType,icon,humidity,summary
16,MAC003449,2012-10-03,30min,"[0.325, 0.8859999999999999, 0.3329999999999999...",24624,Std,ACORN-A,Affluent,block_0,"[NO_Holiday, NO_Holiday, NO_Holiday, NO_Holida...",...,"[219, 219, 223, 223, 224, 224, 223, 223, 222, ...","[14.03, 14.03, 13.1, 13.1, 12.62, 12.62, 12.57...","[11.52, 11.52, 11.82, 11.82, 11.24, 11.24, 10....","[1005.24, 1005.24, 1005.17, 1005.17, 1005.07, ...","[14.03, 14.03, 13.1, 13.1, 12.62, 12.62, 12.57...","[6.62, 6.62, 5.2, 5.2, 4.97, 4.97, 4.51, 4.51,...","[rain, rain, rain, rain, rain, rain, rain, rai...","[partly-cloudy-night, partly-cloudy-night, par...","[0.85, 0.85, 0.92, 0.92, 0.91, 0.91, 0.89, 0.8...","[Partly Cloudy, Partly Cloudy, Partly Cloudy, ..."
57,MAC000768,2012-04-21,30min,"[0.8440000000000001, 0.265, 0.262, 0.233999999...",32544,Std,ACORN-A,Affluent,block_1,"[NO_Holiday, NO_Holiday, NO_Holiday, NO_Holida...",...,"[251, 251, 251, 251, 246, 246, 242, 242, 244, ...","[6.42, 6.42, 6.2, 6.2, 5.68, 5.68, 5.16, 5.16,...","[3.54, 3.54, 3.61, 3.61, 3.52, 3.52, 3.11, 3.1...","[994.96, 994.96, 994.98, 994.98, 994.82, 994.8...","[3.79, 3.79, 3.67, 3.67, 3.15, 3.15, 2.61, 2.6...","[3.64, 3.64, 3.42, 3.42, 3.25, 3.25, 3.13, 3.1...","[rain, rain, rain, rain, rain, rain, rain, rai...","[partly-cloudy-night, partly-cloudy-night, par...","[0.82, 0.82, 0.83, 0.83, 0.86, 0.86, 0.87, 0.8...","[Partly Cloudy, Partly Cloudy, Partly Cloudy, ..."
63,MAC000948,2012-05-02,30min,"[0.008, 0.009, 0.008, 0.008, 0.008, 0.009, 0.0...",32016,Std,ACORN-A,Affluent,block_1,"[NO_Holiday, NO_Holiday, NO_Holiday, NO_Holida...",...,"[351, 351, 0, 0, 0, 0, 351, 351, 348, 348, 3, ...","[11.81, 11.81, 11.12, 11.12, 11.2, 11.2, 11.18...","[10.47, 10.47, 10.15, 10.15, 9.89, 9.89, 9.29,...","[1021.42, 1021.42, 1021.44, 1021.44, 1021.33, ...","[11.81, 11.81, 11.12, 11.12, 11.2, 11.2, 11.18...","[2.53, 2.53, 2.41, 2.41, 2.06, 2.06, 2.98, 2.9...","[rain, rain, rain, rain, rain, rain, rain, rai...","[partly-cloudy-night, partly-cloudy-night, par...","[0.91, 0.91, 0.94, 0.94, 0.92, 0.92, 0.88, 0.8...","[Mostly Cloudy, Mostly Cloudy, Mostly Cloudy, ..."
64,MAC000974,2012-05-03,30min,"[0.163, 0.159, 0.036, 0.032, 0.027999999999999...",31968,Std,ACORN-A,Affluent,block_1,"[NO_Holiday, NO_Holiday, NO_Holiday, NO_Holida...",...,"[11, 11, 16, 16, 25, 25, 16, 16, 28, 28, 29, 2...","[9.17, 9.17, 9.13, 9.13, 9.11, 9.11, 9.04, 9.0...","[7.27, 7.27, 7.6, 7.6, 7.66, 7.66, 7.64, 7.64,...","[1015.07, 1015.07, 1014.57, 1014.57, 1014.08, ...","[7.56, 7.56, 7.73, 7.73, 7.42, 7.42, 7.36, 7.3...","[2.92, 2.92, 2.59, 2.59, 3.03, 3.03, 3.0, 3.0,...","[rain, rain, rain, rain, rain, rain, rain, rai...","[partly-cloudy-night, partly-cloudy-night, par...","[0.88, 0.88, 0.9, 0.9, 0.91, 0.91, 0.91, 0.91,...","[Mostly Cloudy, Mostly Cloudy, Mostly Cloudy, ..."
75,MAC001734,2012-06-02,30min,"[0.187, 0.19, 0.257, 0.291, 0.25, 0.2319999999...",30528,Std,ACORN-A,Affluent,block_1,"[NO_Holiday, NO_Holiday, NO_Holiday, NO_Holida...",...,"[86, 86, 86, 86, 89, 89, 86, 86, 86, 86, 86, 8...","[13.16, 13.16, 12.68, 12.68, 12.61, 12.61, 12....","[12.03, 12.03, 11.6, 11.6, 11.57, 11.57, 11.18...","[1018.05, 1018.05, 1017.88, 1017.88, 1017.65, ...","[13.16, 13.16, 12.68, 12.68, 12.61, 12.61, 12....","[3.54, 3.54, 3.44, 3.44, 3.41, 3.41, 3.14, 3.1...","[rain, rain, rain, rain, rain, rain, rain, rai...","[partly-cloudy-night, partly-cloudy-night, par...","[0.93, 0.93, 0.93, 0.93, 0.93, 0.93, 0.94, 0.9...","[Mostly Cloudy, Mostly Cloudy, Mostly Cloudy, ..."


In [19]:
from src.utils.data_utils import compact_to_expanded

In [20]:
#Converting to expanded form
exp_block_df = compact_to_expanded(block_df, timeseries_col = 'energy_consumption',
static_cols = ["frequency", "series_length", "stdorToU", "Acorn", "Acorn_grouped", "file"],
time_varying_cols = ['holidays', 'visibility', 'windBearing', 'temperature', 'dewPoint',
       'pressure', 'apparentTemperature', 'windSpeed', 'precipType', 'icon',
       'humidity', 'summary'],
ts_identifier = "LCLid")

exp_block_df.head()

100%|██████████| 300/300 [00:02<00:00, 136.44it/s]


,timestamp,LCLid,energy_consumption,frequency,series_length,stdorToU,Acorn,Acorn_grouped,file,holidays,...,windBearing,temperature,dewPoint,pressure,apparentTemperature,windSpeed,precipType,icon,humidity,summary
0,2012-10-03 00:00:00,MAC003449,0.325,30min,24624,Std,ACORN-A,Affluent,block_0,NO_Holiday,...,219,14.03,11.52,1005.24,14.03,6.62,rain,partly-cloudy-night,0.85,Partly Cloudy
1,2012-10-03 00:30:00,MAC003449,0.886,30min,24624,Std,ACORN-A,Affluent,block_0,NO_Holiday,...,219,14.03,11.52,1005.24,14.03,6.62,rain,partly-cloudy-night,0.85,Partly Cloudy
2,2012-10-03 01:00:00,MAC003449,0.333,30min,24624,Std,ACORN-A,Affluent,block_0,NO_Holiday,...,223,13.10,11.82,1005.17,13.10,5.20,rain,partly-cloudy-night,0.92,Partly Cloudy
3,2012-10-03 01:30:00,MAC003449,0.633,30min,24624,Std,ACORN-A,Affluent,block_0,NO_Holiday,...,223,13.10,11.82,1005.17,13.10,5.20,rain,partly-cloudy-night,0.92,Partly Cloudy
4,2012-10-03 02:00:00,MAC003449,0.494,30min,24624,Std,ACORN-A,Affluent,block_0,NO_Holiday,...,224,12.62,11.24,1005.07,12.62,4.97,rain,partly-cloudy-night,0.91,Partly Cloudy


In [22]:
from src.utils.data_utils import reduce_memory_footprint

In [23]:
exp_block_df.info(memory_usage="deep", verbose=False)

<class 'pandas.core.frame.DataFrame'>
Index: 9431808 entries, 0 to 27503
Columns: 21 entries, timestamp to summary
dtypes: datetime64[ns](1), float64(8), int64(2), object(10)
memory usage: 6.6 GB


In [24]:
exp_block_df = reduce_memory_footprint(exp_block_df)

In [25]:
exp_block_df.info(memory_usage="deep", verbose=False)

<class 'pandas.core.frame.DataFrame'>
Index: 9431808 entries, 0 to 27503
Columns: 21 entries, timestamp to summary
dtypes: category(10), datetime64[ns](1), float32(8), int32(2)
memory usage: 602.7 MB


In [27]:
val_mask = (exp_block_df.timestamp.dt.year == 2014) & (exp_block_df.timestamp.dt.month == 1)
test_mask = (exp_block_df.timestamp.dt.year == 2014) & (exp_block_df.timestamp.dt.month == 2)

train = exp_block_df[~(val_mask | test_mask)]
val = exp_block_df[val_mask]
test = exp_block_df[test_mask]
print(f"# of Training samples: {len(train)} | # of Validation samples: {len(val)} | # of Test samples: {len(test)}")
print(f"Max Date in Train: {train.timestamp.max()} | Min Date in Validation: {val.timestamp.min()} | Min Date in Test: {test.timestamp.min()}")


# of Training samples: 8596608 | # of Validation samples: 446400 | # of Test samples: 388800
Max Date in Train: 2013-12-31 23:30:00 | Min Date in Validation: 2014-01-01 00:00:00 | Min Date in Test: 2014-02-01 00:00:00


In [28]:
train.to_parquet("/Users/Soham/Documents/FALL2026/MTFS_textbook_work/data/london_smart_meters/preprocessed/selected_blocks_train.parquet")
val.to_parquet("/Users/Soham/Documents/FALL2026/MTFS_textbook_work/data/london_smart_meters/preprocessed/selected_blocks_val.parquet")
test.to_parquet("/Users/Soham/Documents/FALL2026/MTFS_textbook_work/data/london_smart_meters/preprocessed/selected_blocks_test.parquet")

In [31]:
%pip install sklearn

Defaulting to user installation because normal site-packages is not writeable
    ERROR: Command errored out with exit status 1:
     command: /Library/Developer/CommandLineTools/usr/bin/python3 -c 'import io, os, sys, setuptools, tokenize; sys.argv[0] = '"'"'/private/var/folders/7v/8xmgmgf93dq9mm8nzc14n05w0000gp/T/pip-install-eadehopg/sklearn_9960041918cd49e2bda167a32341229e/setup.py'"'"'; __file__='"'"'/private/var/folders/7v/8xmgmgf93dq9mm8nzc14n05w0000gp/T/pip-install-eadehopg/sklearn_9960041918cd49e2bda167a32341229e/setup.py'"'"';f = getattr(tokenize, '"'"'open'"'"', open)(__file__) if os.path.exists(__file__) else io.StringIO('"'"'from setuptools import setup; setup()'"'"');code = f.read().replace('"'"'\r\n'"'"', '"'"'\n'"'"');f.close();exec(compile(code, __file__, '"'"'exec'"'"'))' egg_info --egg-base /private/var/folders/7v/8xmgmgf93dq9mm8nzc14n05w0000gp/T/pip-pip-egg-info-cjnuwk39
         cwd: /private/var/folders/7v/8xmgmgf93dq9mm8nzc14n05w0000gp/T/pip-install-eadehopg/sklea

In [33]:
from src.imputation.interpolation import SeasonalInterpolation

block_df.energy_consumption = block_df.energy_consumption.progress_apply(lambda x: SeasonalInterpolation(seasonal_period=48*7).fit_transform(x.reshape(-1,1)).squeeze())

  0%|          | 0/300 [00:00<?, ?it/s]/Users/Soham/Library/Python/3.9/lib/python/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/Soham/Library/Python/3.9/lib/python/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/Soham/Library/Python/3.9/lib/python/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/Soham/Library/Python/3.9/lib/python/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/Soham/Library/Python/3.9/lib/python/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_

In [34]:
#Converting to expanded form
exp_block_df = compact_to_expanded(block_df, timeseries_col = 'energy_consumption',
static_cols = ["frequency", "series_length", "stdorToU", "Acorn", "Acorn_grouped", "file"],
time_varying_cols = ['holidays', 'visibility', 'windBearing', 'temperature', 'dewPoint',
       'pressure', 'apparentTemperature', 'windSpeed', 'precipType', 'icon',
       'humidity', 'summary'],
ts_identifier = "LCLid")

exp_block_df.head()

100%|██████████| 300/300 [00:01<00:00, 210.19it/s]


,timestamp,LCLid,energy_consumption,frequency,series_length,stdorToU,Acorn,Acorn_grouped,file,holidays,...,windBearing,temperature,dewPoint,pressure,apparentTemperature,windSpeed,precipType,icon,humidity,summary
0,2012-10-03 00:00:00,MAC003449,0.325,30min,24624,Std,ACORN-A,Affluent,block_0,NO_Holiday,...,219,14.03,11.52,1005.24,14.03,6.62,rain,partly-cloudy-night,0.85,Partly Cloudy
1,2012-10-03 00:30:00,MAC003449,0.886,30min,24624,Std,ACORN-A,Affluent,block_0,NO_Holiday,...,219,14.03,11.52,1005.24,14.03,6.62,rain,partly-cloudy-night,0.85,Partly Cloudy
2,2012-10-03 01:00:00,MAC003449,0.333,30min,24624,Std,ACORN-A,Affluent,block_0,NO_Holiday,...,223,13.10,11.82,1005.17,13.10,5.20,rain,partly-cloudy-night,0.92,Partly Cloudy
3,2012-10-03 01:30:00,MAC003449,0.633,30min,24624,Std,ACORN-A,Affluent,block_0,NO_Holiday,...,223,13.10,11.82,1005.17,13.10,5.20,rain,partly-cloudy-night,0.92,Partly Cloudy
4,2012-10-03 02:00:00,MAC003449,0.494,30min,24624,Std,ACORN-A,Affluent,block_0,NO_Holiday,...,224,12.62,11.24,1005.07,12.62,4.97,rain,partly-cloudy-night,0.91,Partly Cloudy


In [35]:
from src.utils.data_utils import reduce_memory_footprint

In [36]:
exp_block_df.info(memory_usage="deep", verbose=False)

<class 'pandas.core.frame.DataFrame'>
Index: 9431808 entries, 0 to 27503
Columns: 21 entries, timestamp to summary
dtypes: datetime64[ns](1), float64(8), int64(2), object(10)
memory usage: 6.6 GB


In [37]:
exp_block_df = reduce_memory_footprint(exp_block_df)

In [38]:
exp_block_df.info(memory_usage="deep", verbose=False)

<class 'pandas.core.frame.DataFrame'>
Index: 9431808 entries, 0 to 27503
Columns: 21 entries, timestamp to summary
dtypes: category(10), datetime64[ns](1), float32(8), int32(2)
memory usage: 602.7 MB


In [39]:
test_mask = (exp_block_df.timestamp.dt.year==2014) & (exp_block_df.timestamp.dt.month==2)
val_mask = (exp_block_df.timestamp.dt.year==2014) & (exp_block_df.timestamp.dt.month==1)

train = exp_block_df[~(val_mask|test_mask)]
val = exp_block_df[val_mask]
test = exp_block_df[test_mask]
print(f"# of Training samples: {len(train)} | # of Validation samples: {len(val)} | # of Test samples: {len(test)}")
print(f"Max Date in Train: {train.timestamp.max()} | Min Date in Validation: {val.timestamp.min()} | Min Date in Test: {test.timestamp.min()}")

# of Training samples: 8596608 | # of Validation samples: 446400 | # of Test samples: 388800
Max Date in Train: 2013-12-31 23:30:00 | Min Date in Validation: 2014-01-01 00:00:00 | Min Date in Test: 2014-02-01 00:00:00
